# Middleware

Summarization Middleware

based on Message Length

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

# building the agent
agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [5]:
# run with thread id 
config={"configurable":{"thread_id":"test-1"}}

In [6]:
# test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Message: {response}")
    print(f"Messages: {len(response['messages'])}")

Message: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='0a1efa52-d0d0-4e47-8ade-440c71016d70'), AIMessage(content='<think>\nOkay, let me think about this. The question is asking what 2 plus 2 is. Hmm, that seems straightforward, but I want to make sure I\'m not missing anything. Let me start by recalling basic addition. When you add two numbers together, you\'re combining their quantities. So 2 and 2 are both the number two. If I have two apples and then get two more apples, how many do I have in total? That should be four apples. \n\nWait, maybe I should break it down further. Addition is one of the fundamental operations in arithmetic. The plus sign (+) represents the operation of adding. So 2 + 2 means starting at 2 and then moving two units forward on the number line. Starting at 2, moving one unit gets me to 3, and another unit gets me to 4. So that\'s 4. \n\nAlternatively, using physical objects: if I have two pencils and someon

based on Token Size

In [1]:
# libraries 
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

building a simple tool

In [2]:
@tool
def search_hotel(city: str) -> str:
    """Search hotels - retruns long response to use more tokens"""
    return f"""Hotels in {city}
    1. Chiku Hotel - 5 star, 10k INR/night, spa, pool, gym
    2. Bhala Hotel - 7 star 70k INR/night, turf, gym, basketball court
    3. Pot Hotel - 1 star, 2k INR/night, bed"""

building the agent


In [4]:
agent  = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="groq:llama-3.1-8b-instant",
        trigger=("tokens",550),
        keep=("tokens",200)
    ),
    ]
)

config = {"configurable": {"thread_id":"test-1"}}

# token counter
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars

testing out the agent

In [5]:
cities = ["Paris","London","Tokyo","New York","Dubai","Singapore"]
for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: {tokens} tokens, {len(response["messages"])} messages")
    print(f"{response["messages"]}")

Paris: 254 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='41c590c4-3285-4a59-9dcb-992ebe408d78'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'a9b8q5n2t', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotel'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 228, 'total_tokens': 243, 'completion_time': 0.023693415, 'completion_tokens_details': None, 'prompt_time': 0.014366293, 'prompt_tokens_details': None, 'queue_time': 0.046157143, 'total_time': 0.038059708}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f172e-ce4b-7080-9ea9-7c15e9576f07-0', tool_calls=[{'name': 'search_hotel', 'args': {'city': 'Paris'}, 'id': 'a9b8q5n2t', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'i

based on Fraction

In [6]:
agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",
            trigger=("fraction",0.005),
            keep=("fraction",0.002)
        )
    ]
)
config = {"configurable":{"thread_id":"test-1"}}

def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

In [7]:
cities = ["Paris","London","Tokyo","New York","Dubai","Singapore"]
for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    fraction = tokens/131072
    print(f"{city}: {tokens} tokens ({fraction:.4%}), {len(response["messages"])} messages")
    print(f"{response["messages"]}")

Paris: 67 tokens (0.0511%), 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='637a1a95-3d3f-4ec7-b438-2acb6c49024b'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'j4g6d8mfe', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotel'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 228, 'total_tokens': 243, 'completion_time': 0.026753025, 'completion_tokens_details': None, 'prompt_time': 0.016644811, 'prompt_tokens_details': None, 'queue_time': 0.047275069, 'total_time': 0.043397836}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f1736-3a68-7822-9833-05a1673d65d2-0', tool_calls=[{'name': 'search_hotel', 'args': {'city': 'Paris'}, 'id': 'j4g6d8mfe', 'type': 'tool_call'}], invalid_tool_calls=[], usage_met

# Human in the Loop Middleware

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

sending and reading email agent

tools for sending and reading email sample